# MemRec-TTM: experimental results

This notebook recomputes the results tables of the master's thesis *Applying Retrieval-Augmented Generation to
Recommender Systems* from the saved per-user predictions (`results/runs/<run>/predictions.jsonl`). It makes no LLM
call and needs no GPU; a full rerun takes about a minute:

```bash
pip install numpy pandas jupyter
jupyter nbconvert --to notebook --execute --inplace notebooks/results.ipynb
```

**Protocol.** Two datasets with timestamps: Amazon Books (the Kaggle *Amazon Books Reviews* dataset, derived from the
Amazon data of McAuley et al., 1996–2013; sparse, text-rich) and MovieLens-32M (dense, little text; interactions are
ratings of 4 stars or more). Data are split at a global time cutoff, and everything a model uses lies before it. Each
test user has 10 candidates: the target item (the first interaction after the cutoff) and 9 negatives drawn from the
same popularity bin (`popmatched`). The test set has 1,000 users per dataset (the original 300 plus 700 additional
users). The main metric is NDCG@5. 95% confidence intervals of paired differences come from a stratified bootstrap
(10,000 resamples of users within each dataset × part stratum); `*` marks an interval that excludes 0.

**Configurations.**

| Name | Description |
|---|---|
| Vanilla LLM | The LLM ranks the 10 candidates from the user's recent history, without memory |
| MemRec | The original MemRec framework (Chen et al., ACL 2026): graph-structured collaborative memory, LM_Mem summarizes it into facets (Stage-R), LLM_Rec scores the candidates (Stage-ReRank), Stage-W writes memory |
| C0 | *Recent trajectory* retrieval: the 10 items the user interacted with most recently, with title and short description |
| C3 | *Transition evidence* retrieval: PPR from the 6 most recent items on a directed, time-ordered transition graph; each candidate receives one evidence sentence |
| **MemRec-TTM** | *MemRec with Temporal Transition Memory* = MemRec + C0 + C3 in the Stage-ReRank prompt; the rest of MemRec is unchanged |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))
import analyze_test1000 as A  # noqa: E402  (run definitions and per-user scores)
from src.eval.stats import load_predictions, ndcg_at, stratified_bootstrap  # noqa: E402

pd.set_option('display.max_colwidth', 80)
DS = {'amz': 'Amazon Books', 'ml': 'MovieLens-32M'}


def absolute(arm, d):
    # metrics on the 1,000 test users (300 + 700); two-seed arms are averaged per user
    us = A.common_users([arm], d)
    if us is None:
        return None
    out = {}
    for m, key in (('hit1', 'Hit@1'), ('hit3', 'Hit@3'), ('ndcg3', 'NDCG@3'), ('hit5', 'Hit@5'), ('ndcg5', 'NDCG@5')):
        v = A.scores(arm, d, us, m)
        out[key] = float(np.concatenate([v['t300'], v['x700']]).mean())
    out['n'] = sum(len(x) for x in us.values())
    return out


def fmt_delta(r):
    star = '*' if r['ci_low'] > 0 or r['ci_high'] < 0 else ''
    return f"{r['delta']:+.3f} [{r['ci_low']:+.3f}; {r['ci_high']:+.3f}]{star}"


def delta(a, b):
    # B - A, NDCG@5, paired by user: per dataset and pooled over both datasets
    out, pooled = {}, []
    for d in DS:
        us = A.common_users([a, b], d)
        sa, sb = A.scores(a, d, us, 'ndcg5'), A.scores(b, d, us, 'ndcg5')
        parts = [sb[p] - sa[p] for p in ('t300', 'x700')]
        out[DS[d]] = fmt_delta(stratified_bootstrap(parts))
        pooled += parts
    out['Pooled'] = fmt_delta(stratified_bootstrap(pooled))
    return out

## 1. Comparison with related methods (Amazon Books and MovieLens-32M)

Methods are grouped by the taxonomy of the MemRec paper (its *Baselines* section): no LLM; LLM without explicit
memory; static memory (iAgent); dynamic isolated memory (i²Agent, AgentCF, RecBot); dynamic collaborative memory
(MemRec). MemRec-TTM belongs to the last group: it keeps MemRec's collaborative memory and adds two time-aware
retrieval sources (C0, C3). PPR on the transition graph is the C3 source used alone, without an LLM.

iAgent, i²Agent and RecBot need natural-language user instructions (available only in InstructRec), and AgentCF
optimizes user and item agents with an LLM over all training interactions, so they are not run as such on these two
datasets. To represent these two memory groups under identical conditions (same LLM, prompts and candidates), the
thesis builds two variants inside the MemRec pipeline, marked (\*). Only the memory changes:

| | Neighbors read by Stage-R | Memory writes (Stage-W, one warm-up round) |
|---|---|---|
| MemRec | other users with co-interactions + the user's own items | user, clicked item, propagated to up to 8 neighbors |
| MemRec w/o collaborative neighbors (≈ dynamic isolated memory: AgentCF, i²Agent) | the user's own items only | user and clicked item, no propagation |
| MemRec w/o collaborative neighbors and memory writes (≈ static memory: iAgent) | the user's own items only | none (no warm-up) |

They are stand-ins for the memory types, not re-implementations of the original agents. Section 2 places the original
agents relative to MemRec on MemRec's own benchmark.

In [2]:
G = 'Gemini 3.5 Flash-Lite'
NO_COLLAB = 'MemRec w/o collaborative neighbors (*)'
NO_COLLAB_WRITE = 'MemRec w/o collaborative neighbors and memory writes (*)'
ROWS = [  # (group, method, LLM_Rec, arm); (*) = stand-in built in MemRec, (†) = thesis control configuration
    ('No LLM', 'Random', '-', 'random'),
    ('No LLM', 'Pop (popularity)', '-', 'pop'),
    ('No LLM', 'BPR-MF', '-', 'bprmf'),
    ('No LLM', 'LightGCN', '-', 'lightgcn'),
    ('No LLM', 'SASRec (BCE, as in the original paper)', '-', 'sasrec'),
    ('No LLM', 'SASRec (cross-entropy)', '-', 'sasrecce'),
    ('No LLM', 'PPR on the transition graph', '-', 'ppr'),
    ('LLM, no explicit memory', 'Vanilla LLM', G, 'vanilla'),
    ('LLM, no explicit memory', 'Vanilla LLM + C3 (†)', G, 'vanilla_evid'),
    ('LLM, static memory', NO_COLLAB_WRITE, G, 'static'),
    ('LLM, dynamic isolated memory', NO_COLLAB, G, 'isolated'),
    ('LLM, dynamic collaborative memory', 'MemRec', G, 'memrec'),
    ('LLM, dynamic collaborative memory', 'MemRec-TTM (proposed)', G, 'ttm'),
    ('LLM, dynamic collaborative memory', 'MemRec-TTM w/o Stage-R (†)', G, 'nofacet'),
    ('LLM, dynamic collaborative memory', 'MemRec-TTM (proposed)', 'Qwen3-8B, zero-shot', 'q_ttm_rank'),
    ('LLM, dynamic collaborative memory', 'MemRec-TTM (proposed)', 'Qwen3-8B + GRPO', 'grpo_avg'),
    ('LLM, dynamic collaborative memory', 'MemRec-TTM (proposed)', 'Qwen3-8B + Rank-GRPO', 'rgrpo_avg'),
]
METRICS = ('Hit@1', 'Hit@3', 'NDCG@3', 'Hit@5', 'NDCG@5')
table = []
for group, name, llm, arm in ROWS:
    row = {'Group': group, 'Method': name, 'LLM_Rec': llm}
    for d, dn in DS.items():
        r = absolute(arm, d)
        for k in METRICS:
            row[(dn, k)] = r[k] if r else np.nan
    table.append(row)
t1 = pd.DataFrame(table).set_index(['Group', 'Method', 'LLM_Rec'])
t1.columns = pd.MultiIndex.from_tuples(t1.columns)
print('Test set, 1,000 users per dataset, 10 candidates (popmatched). Qwen3-8B + RL: mean of two seeds (42, 7).\n'
      '(*) stand-in for a memory group, built inside MemRec; (†) control configuration of the thesis.')
t1.style.format('{:.3f}').highlight_max(axis=0, props='font-weight: bold')

Test set, 1,000 users per dataset, 10 candidates (popmatched). Qwen3-8B + RL: mean of two seeds (42, 7).
(*) stand-in for a memory group, built inside MemRec; (†) control configuration of the thesis.


NDCG@5 difference to MemRec (paired by user, 95% confidence interval):

In [3]:
pd.DataFrame({name + ('' if llm == '-' else f' ({llm})'): delta('memrec', arm)
              for _, name, llm, arm in ROWS if arm != 'memrec'}).T

,Amazon Books,MovieLens-32M,Pooled
Random,-0.311 [-0.341; -0.279]*,-0.243 [-0.274; -0.213]*,-0.277 [-0.299; -0.254]*
Pop (popularity),-0.223 [-0.255; -0.190]*,-0.200 [-0.234; -0.167]*,-0.212 [-0.235; -0.188]*
BPR-MF,-0.201 [-0.232; -0.170]*,+0.084 [+0.056; +0.112]*,-0.058 [-0.078; -0.037]*
LightGCN,-0.043 [-0.072; -0.014]*,+0.074 [+0.045; +0.102]*,+0.015 [-0.005; +0.036]
"SASRec (BCE, as in the original paper)",-0.285 [-0.316; -0.253]*,-0.002 [-0.034; +0.028],-0.143 [-0.165; -0.121]*
SASRec (cross-entropy),-0.274 [-0.306; -0.241]*,+0.049 [+0.021; +0.078]*,-0.112 [-0.134; -0.091]*
PPR on the transition graph,-0.063 [-0.092; -0.033]*,+0.030 [+0.001; +0.060]*,-0.016 [-0.037; +0.005]
Vanilla LLM (Gemini 3.5 Flash-Lite),+0.012 [-0.006; +0.030],+0.083 [+0.063; +0.103]*,+0.047 [+0.034; +0.061]*
Vanilla LLM + C3 (†) (Gemini 3.5 Flash-Lite),+0.021 [+0.003; +0.040]*,+0.101 [+0.081; +0.121]*,+0.061 [+0.048; +0.075]*
MemRec w/o collaborative neighbors and memory writes (*) (Gemini 3.5 Flash-Lite),-0.002 [-0.019; +0.015],+0.027 [+0.008; +0.047]*,+0.013 [-0.000; +0.025]


NDCG@5 difference of MemRec-TTM (Gemini) to the representative methods of each group:

In [4]:
pd.DataFrame({f'MemRec-TTM − {name}': delta(arm, 'ttm')
              for name, arm in (('BPR-MF', 'bprmf'), ('LightGCN', 'lightgcn'), ('SASRec (cross-entropy)', 'sasrecce'),
                                ('PPR', 'ppr'), ('Vanilla LLM', 'vanilla'), (NO_COLLAB_WRITE, 'static'),
                                (NO_COLLAB, 'isolated'), ('MemRec', 'memrec'))}).T

,Amazon Books,MovieLens-32M,Pooled
MemRec-TTM − BPR-MF,+0.223 [+0.193; +0.253]*,-0.007 [-0.034; +0.020],+0.108 [+0.087; +0.128]*
MemRec-TTM − LightGCN,+0.065 [+0.036; +0.094]*,+0.003 [-0.024; +0.031],+0.034 [+0.014; +0.054]*
MemRec-TTM − SASRec (cross-entropy),+0.296 [+0.263; +0.329]*,+0.028 [+0.000; +0.055]*,+0.162 [+0.141; +0.183]*
MemRec-TTM − PPR,+0.085 [+0.057; +0.113]*,+0.047 [+0.019; +0.074]*,+0.066 [+0.047; +0.085]*
MemRec-TTM − Vanilla LLM,+0.010 [-0.006; +0.027],-0.006 [-0.023; +0.011],+0.002 [-0.009; +0.014]
MemRec-TTM − MemRec w/o collaborative neighbors and memory writes (*),+0.024 [+0.007; +0.042]*,+0.050 [+0.031; +0.068]*,+0.037 [+0.024; +0.050]*
MemRec-TTM − MemRec w/o collaborative neighbors (*),+0.005 [-0.012; +0.022],+0.051 [+0.033; +0.069]*,+0.028 [+0.016; +0.040]*
MemRec-TTM − MemRec,+0.022 [+0.009; +0.035]*,+0.077 [+0.062; +0.092]*,+0.049 [+0.040; +0.059]*


## 2. MemRec's original benchmark (InstructRec Books)

The upper part of the table lists the numbers reported in the MemRec paper (Chen et al., 2026, main results table,
*Books*: gpt-4o-mini, full test set, 10 candidates). The lower part lists the thesis's reruns on the first 300 users
of a seeded (42) permutation of the 1,000-user sample published by MemRec, with the original candidate lists:
traditional baselines are trained on the same leave-one-out split, and LLM configurations use Gemini 3.5 Flash-Lite
with the user instructions, like the original MemRec. The LLM methods in the two parts differ in LLM and sample size,
so only their order within one part is comparable; the traditional baselines do not depend on the LLM and can be
compared more directly.

On this benchmark the interaction order is almost alphabetical by ASIN (88.6% of adjacent item pairs have increasing
ASINs), so the temporal signal is weak; the benchmark serves as a control for that case.

In [5]:
PAPER_BOOKS = [  # (group in the MemRec paper, method, H@1, H@3, N@3, H@5, N@5)
    ('Traditional', 'LightGCN', 0.1753, 0.3259, 0.2596, 0.5703, 0.3592),
    ('Traditional', 'SASRec', 0.0914, 0.2830, 0.2001, 0.4845, 0.2824),
    ('Traditional', 'P5', 0.2192, 0.3607, 0.2994, 0.5273, 0.3671),
    ('No explicit memory', 'Vanilla LLM', 0.3138, 0.5617, 0.4533, 0.7270, 0.5226),
    ('Static memory', 'iAgent', 0.3925, 0.5560, 0.4858, 0.6905, 0.5409),
    ('Dynamic isolated memory', 'RecBot', 0.3984, 0.5491, 0.4846, 0.6786, 0.5376),
    ('Dynamic isolated memory', 'AgentCF', 0.3457, 0.6060, 0.4960, 0.7403, 0.5512),
    ('Dynamic isolated memory', 'i²Agent', 0.4453, 0.6517, 0.5649, 0.7708, 0.6138),
    ('Dynamic collaborative memory', 'MemRec', 0.5117, 0.6915, 0.6152, 0.8007, 0.6601),
]
cols = ['Hit@1', 'Hit@3', 'NDCG@3', 'Hit@5', 'NDCG@5']
rows = [('MemRec paper (gpt-4o-mini, full test set)', g, m, *v) for g, m, *v in PAPER_BOOKS]

OURS_BOOKS = [  # reruns on the 300 test users
    ('Traditional', 'Pop', 'books_pop'),
    ('Traditional', 'BPR-MF', 'books_bprmf'),
    ('Traditional', 'LightGCN', 'books_lightgcn'),
    ('Traditional', 'SASRec (BCE)', 'books_sasrec'),
    ('Traditional', 'SASRec (cross-entropy)', 'books_sasrecce'),
    ('No explicit memory', 'Vanilla LLM', 'books_vanilla_t300'),
    ('No explicit memory', 'Vanilla LLM + C3', 'books_vanilla_evid_t300'),
    ('Static memory', NO_COLLAB_WRITE, 'books_static_t300'),
    ('Dynamic isolated memory', NO_COLLAB, 'books_isolated_t300'),
    ('Dynamic collaborative memory', 'MemRec', 'books_memrec_t300'),
    ('Dynamic collaborative memory', 'MemRec-TTM (proposed)', 'books_ttm_evid_recent_t300'),
]
for g, m, run in OURS_BOOKS:
    recs = load_predictions(str(ROOT / 'results' / 'runs' / run / 'predictions.jsonl'))
    p = np.array([r['position'] for r in recs.values() if r.get('split', 'test') == 'test'])
    rows.append(('Thesis reruns (300 users; LLM: Gemini 3.5 Flash-Lite)', g, m,
                 (p == 0).mean(), (p < 3).mean(), np.mean([ndcg_at(x, 3) for x in p]),
                 (p < 5).mean(), np.mean([ndcg_at(x, 5) for x in p])))
t2 = pd.DataFrame(rows, columns=['Source', 'Group', 'Method'] + cols).set_index(['Source', 'Group', 'Method'])
t2.style.format('{:.3f}')

In [6]:
# paired NDCG@5 differences on InstructRec Books (300 users)
RUN_OF = {m: r for _, m, r in OURS_BOOKS}


def books_delta(a, b):
    ra = load_predictions(str(ROOT / 'results' / 'runs' / RUN_OF[a] / 'predictions.jsonl'))
    rb = load_predictions(str(ROOT / 'results' / 'runs' / RUN_OF[b] / 'predictions.jsonl'))
    us = sorted(set(ra) & set(rb))
    return fmt_delta(stratified_bootstrap([np.array([ndcg_at(rb[u]['position'], 5) - ndcg_at(ra[u]['position'], 5)
                                                     for u in us])]))


pd.Series({'MemRec − Vanilla LLM': books_delta('Vanilla LLM', 'MemRec'),
           f'{NO_COLLAB_WRITE} − MemRec': books_delta('MemRec', NO_COLLAB_WRITE),
           f'{NO_COLLAB} − MemRec': books_delta('MemRec', NO_COLLAB),
           'MemRec-TTM − MemRec': books_delta('MemRec', 'MemRec-TTM (proposed)'),
           f'MemRec-TTM − {NO_COLLAB}': books_delta(NO_COLLAB, 'MemRec-TTM (proposed)')},
          name='Δ NDCG@5 [95% CI]').to_frame()

,Δ NDCG@5 [95% CI]
MemRec − Vanilla LLM,+0.022 [-0.011; +0.054]
MemRec w/o collaborative neighbors and memory writes (*) − MemRec,+0.020 [-0.004; +0.045]
MemRec w/o collaborative neighbors (*) − MemRec,+0.043 [+0.016; +0.071]*
MemRec-TTM − MemRec,+0.016 [-0.007; +0.040]
MemRec-TTM − MemRec w/o collaborative neighbors (*),-0.027 [-0.051; -0.002]*


## 3. Contribution of each retrieval source (validation and test pooled)

The differences below pool the 1,000-user test set with the validation sets of both datasets (2,200 users; rows
marked [val2] also include val2, 2,800 users). Each row changes exactly one component of the Stage-ReRank prompt.

In [7]:
POOLED_ROWS = [  # (description, A, B, use val2): difference = B - A
    ('MemRec − Vanilla LLM', 'vanilla', 'memrec', False),
    ('Adding C0 to MemRec', 'memrec', 'memrec_recent', False),
    ('Adding C3 to MemRec + C0 (= MemRec-TTM)', 'memrec_recent', 'ttm', False),
    ('MemRec-TTM − MemRec', 'memrec', 'ttm', False),
    ('MemRec-TTM − (Vanilla LLM + C3) [val2]', 'vanilla_evid', 'ttm', True),
    ('Removing Stage-R (facets) from MemRec-TTM [val2]', 'ttm', 'nofacet', True),
]
rows = []
for label, a, b, v2 in POOLED_ROWS:
    r = A.pooled_delta(a, b, v2)
    rows.append({'Comparison': label, 'Δ NDCG@5 [95% CI]': fmt_delta(r), 'Users': r['n']})
pd.DataFrame(rows).set_index('Comparison')

,Δ NDCG@5 [95% CI],Users
Comparison,,
MemRec − Vanilla LLM,-0.048 [-0.060; -0.035]*,2200
Adding C0 to MemRec,+0.041 [+0.033; +0.050]*,2200
Adding C3 to MemRec + C0 (= MemRec-TTM),+0.011 [+0.004; +0.018]*,2200
MemRec-TTM − MemRec,+0.052 [+0.043; +0.062]*,2200
MemRec-TTM − (Vanilla LLM + C3) [val2],-0.008 [-0.017; +0.002],2800
Removing Stage-R (facets) from MemRec-TTM [val2],+0.010 [+0.001; +0.019]*,2800


## 4. Reinforcement learning for the ranking step (LLM_Rec = Qwen3-8B)

LLM_Rec is trained with GRPO and Rank-GRPO (LoRA r = 16, 500 steps, about 4.6 hours on one H100 per run), two seeds
per algorithm (42 and 7). LM_Mem and the memory stay frozen; the prompt is the MemRec-TTM Stage-ReRank prompt with a
ranked-list output. Training data come from a separate *training world* (section 5).

In [8]:
RL_ROWS = [
    ('Qwen3-8B, zero-shot', 'q_ttm_rank'),
    ('Qwen3-8B + GRPO, seed 42', 'ttm_grpo'), ('Qwen3-8B + GRPO, seed 7', 'ttm_grpo_s2'),
    ('Qwen3-8B + GRPO, mean of two seeds', 'grpo_avg'),
    ('Qwen3-8B + Rank-GRPO, seed 42', 'ttm_rgrpo2'), ('Qwen3-8B + Rank-GRPO, seed 7', 'ttm_rgrpo_s2'),
    ('Qwen3-8B + Rank-GRPO, mean of two seeds', 'rgrpo_avg'),
    ('Gemini 3.5 Flash-Lite (reference)', 'ttm'),
]
t4 = pd.DataFrame({label: {(DS[d], k): absolute(arm, d)[k] for d in DS for k in ('NDCG@5', 'Hit@1')}
                   for label, arm in RL_ROWS}).T
t4.columns = pd.MultiIndex.from_tuples(t4.columns)
t4.style.format('{:.3f}')

In [9]:
pd.DataFrame({
    'Rank-GRPO − GRPO (mean of two seeds)': delta('grpo_avg', 'rgrpo_avg'),
    'Rank-GRPO − zero-shot': delta('q_ttm_rank', 'rgrpo_avg'),
    'GRPO − zero-shot': delta('q_ttm_rank', 'grpo_avg'),
    'Rank-GRPO (Qwen3-8B) − Gemini 3.5 Flash-Lite': delta('ttm', 'rgrpo_avg'),
}).T

,Amazon Books,MovieLens-32M,Pooled
Rank-GRPO − GRPO (mean of two seeds),+0.014 [+0.003; +0.026]*,+0.031 [+0.018; +0.044]*,+0.022 [+0.014; +0.031]*
Rank-GRPO − zero-shot,+0.077 [+0.058; +0.096]*,+0.137 [+0.115; +0.160]*,+0.107 [+0.093; +0.122]*
GRPO − zero-shot,+0.063 [+0.046; +0.080]*,+0.107 [+0.088; +0.126]*,+0.085 [+0.072; +0.098]*
Rank-GRPO (Qwen3-8B) − Gemini 3.5 Flash-Lite,-0.037 [-0.057; -0.017]*,+0.032 [+0.010; +0.053]*,-0.003 [-0.017; +0.012]


## 5. Data used for reinforcement learning

The original MemRec and MemRec-TTM with Gemini update no parameters. Reinforcement learning needs additional labeled
data but no external knowledge: the data are built from the same interaction logs, with the same protocol and an
earlier cutoff, so nothing from the test period leaks. The table below is read directly from the cohort files.

In [10]:
import json
from datetime import datetime, timezone

rows = []
for d, name in (('temporal-amazon-books', 'Amazon Books'), ('temporal-ml32m', 'MovieLens-32M')):
    for role, f in (('RL training world', f'{d}-train08_popmatched_t2000_v200.json'),
                    ('Test (original 300 users)', f'{d}_popmatched_t300_v100.json'),
                    ('Test (additional 700 users)', f'{d}-ext_popmatched_t700_v0.json')):
        c = json.load(open(ROOT / 'data' / 'cohorts' / f))
        train = 'train08' in f
        rows.append({'Dataset': name, 'Role': role,
                     'Cutoff (time quantile)': c['cutoff_q'],
                     'Cutoff (date)': datetime.fromtimestamp(c['cutoff'], timezone.utc).date().isoformat(),
                     'Label': 'first interaction in [0.8, 0.9)' if train else 'first interaction after 0.9',
                     'Users': f"{c['n_test']} training + {c['n_val']} validation" if train else f"{c['n_test']} test",
                     'Candidates per user': c['n_candidates']})
pd.DataFrame(rows).set_index(['Dataset', 'Role'])

Cutoff (time quantile)  \
Dataset       Role                                                  
Amazon Books  RL training world                               0.8   
              Test (original 300 users)                       0.9   
              Test (additional 700 users)                     0.9   
MovieLens-32M RL training world                               0.8   
              Test (original 300 users)                       0.9   
              Test (additional 700 users)                     0.9   

                                          Cutoff (date)  \
Dataset       Role                                        
Amazon Books  RL training world              2011-09-16   
              Test (original 300 users)      2012-11-07   
              Test (additional 700 users)    2012-11-07   
MovieLens-32M RL training world              2018-09-28   
              Test (original 300 users)      2020-10-22   
              Test (additional 700 users)    2020-10-22   

                                                                     Label  \
Dataset       Role                                                           
Amazon Books  RL training world            first interaction in [0.8, 0.9)   
              Test (original 300 users)        first interaction after 0.9   
              Test (additional 700 users)      first interaction after 0.9   
MovieLens-32M RL training world            first interaction in [0.8, 0.9)   
              Test (original 300 users)        first interaction after 0.9   
              Test (additional 700 users)      first interaction after 0.9   

                                                                    Users  \
Dataset       Role                                                          
Amazon Books  RL training world            2000 training + 200 validation   
              Test (original 300 users)                          300 test   
              Test (additional 700 users)                        700 test   
MovieLens-32M RL training world            2000 training + 200 validation   
              Test (original 300 users)                          300 test   
              Test (additional 700 users)                        700 test   

                                           Candidates per user  
Dataset       Role                                              
Amazon Books  RL training world                             10  
              Test (original 300 users)                     10  
              Test (additional 700 users)                   10  
MovieLens-32M RL training world                             10  
              Test (original 300 users)                     10  
              Test (additional 700 users)                   10

Labels of the training world are the first interaction after the 0.8 time quantile and before the 0.9 quantile, i.e.
entirely before the test period (cutoff 0.9). Users of the test and validation cohorts are excluded from the training
world, and the additional test users exclude the training-world users. Compared with the original MemRec,
reinforcement learning therefore needs: (1) 2,000 labeled training prompts and 200 validation prompts per dataset,
built from the existing interaction logs; (2) compute for training (one H100 GPU, about 4.6 hours per run); (3) an
open-weight LLM whose parameters can be updated (Qwen3-8B). No metadata or knowledge source beyond what MemRec-TTM
already uses is involved.